# Portefeuille ESG ERC 2°C Aligned

**Auteur**: BLACKBOXAI | **Date**: 2024

Conversion du script `portfolio_esg.py` en notebook interactif.

## Objectif
Construire un portefeuille ESG aligné 2°C avec stratégie **Equal Risk Contribution (ERC)** + optimisation rendements attendus.

**Étapes**:
1. ❌ Chargement données Parquet
2. ✂️ Exclusion 30% poids les plus bas ESG **par secteur**
3. 📈 Rendements historiques + covariance (1 an)
4. 🔧 **Optimisation ERC** (ITR ≤ 2°C, ∑w=1, w∈[0,1])
5. 📊 Visualisations + sauvegarde

**Sorties**: `portfolio_final.csv` + `portfolio_visualization.html`

In [2]:
import pandas as pd
import numpy as np
from scipy.optimize import minimize
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.express as px
from pathlib import Path

print('✅ Imports OK | Plotly pour viz interactives')

✅ Imports OK | Plotly pour viz interactives


## 1. CHARGEMENT DONNÉES (Ethics_data)

In [ ]:
print('📊 Chargement...')

# Chemins (relatif au notebook)
data_path = Path('.') / 'data'

# Données
metadata = pd.read_parquet(data_path / 'metadata.parquet',engine='pyarrow')
print(f'   Metadata: {metadata.shape} | Secteurs: {metadata.SECTOR.nunique()}')

universe = pd.read_parquet(data_path / 'universe.parquet',engine='pyarrow').T
esg_scores = pd.read_parquet(data_path / 'esg_score.parquet',engine='pyarrow').T
itr = pd.read_parquet(data_path / 'itr.parquet',engine='pyarrow').T

latest_date = universe.columns[-1]
print(f'   Date: {latest_date}')

# Latest
weights = universe[latest_date].dropna() * 100
esg_latest = esg_scores[latest_date].dropna()
itr_latest = itr[latest_date].dropna()

# DF fusionné
df = pd.DataFrame({
    'ticker': weights.index,
    'weight': weights.values,
    'esg': esg_latest.reindex(weights.index).values,
    'itr': itr_latest.reindex(weights.index).values
}).join(metadata.set_index('ID')[['SECTOR', 'NAME', 'COUNTRY']], on='ticker')

df = df.dropna()
print(f'   Univers initial: {len(df)} tickers')
df.head()

📊 Chargement...


ArrowKeyError: No type extension with name arrow.py_extension_type found

## 2. EXCLUSION 30% PLUS BAS ESG **PAR SECTEUR**

In [ ]:
print('✂️ Exclusion par secteur...')

eligible_tickers = []
sector_total_weight = df.groupby('SECTOR')['weight'].sum()

for sector, group in df.groupby('SECTOR'):
    sector_weight = sector_total_weight[sector]
    exclude_weight = 0.3 * sector_weight
    
    sorted_group = group.sort_values('esg')  # Plus bas ESG first
    cum_weight = sorted_group['weight'].cumsum()
    keep_mask = cum_weight > exclude_weight
    eligible = sorted_group[keep_mask]
    
    eligible_tickers.append(eligible)
    print(f'   {sector}: {len(group)} → {len(eligible)}')

df_eligible = pd.concat(eligible_tickers, ignore_index=True)
print(f'✅ Univers éligible: {len(df_eligible)}')
df_eligible.head()

## 3. RENDEMENTS HISTORIQUES (252 jours ~1 an)

In [ ]:
print('📈 Rendements...')

prices = pd.read_parquet(data_path / 'price.parquet')
tickers_price = prices.columns.intersection(df_eligible.ticker)

lookback = 252
recent_prices = prices[tickers_price].iloc[-lookback:].dropna(axis=1, how='all')
returns = recent_prices.pct_change().dropna()

mu = returns.mean() * 252  # Annualisé
cov = returns.cov() * 252

valid_idx = df_eligible.ticker.isin(returns.columns)
df_portfolio = df_eligible[valid_idx].copy()
df_portfolio['mu'] = mu.reindex(df_portfolio.ticker).values * 100

n_assets = len(df_portfolio)
print(f'✅ {n_assets} tickers OK')

tickers_opt = df_portfolio.ticker.values
itr_opt = df_portfolio.itr.values
w0 = np.ones(n_assets) / n_assets

df_portfolio[['mu', 'esg', 'itr']].describe()

## 4. OPTIMISATION **ERC** + ITR ≤ 2°C

In [ ]:
print('🔧 Optimisation...')

def portfolio_risk(w, cov): return np.sqrt(w.T @ cov @ w)
def risk_contribution(w, cov):
    vol_port = portfolio_risk(w, cov)
    return w * (cov @ w / vol_port)
def erc_objective(w, cov):
    contrib = risk_contribution(w, cov)
    target = 1.0 / len(w)
    return np.sum((contrib - target)**2)
def itr_constraint(w): return np.sum(w * itr_opt) - 2.0

constraints = [
    {'type': 'eq', 'fun': lambda w: np.sum(w) - 1},
    {'type': 'ineq', 'fun': itr_constraint}
]
bounds = [(0, 1) for _ in range(n_assets)]

res = minimize(erc_objective, w0, args=(cov.values,), method='SLSQP',
               bounds=bounds, constraints=constraints, options={'maxiter': 1000})

if res.success:
    weights_opt = res.x
else:
    print('⚠️ Using uniform')
    weights_opt = w0

df_results = df_portfolio.copy()
df_results['weight_opt'] = weights_opt
df_results['risk_contrib'] = risk_contribution(weights_opt, cov.values)

# Métriques
port_itr = np.sum(weights_opt * itr_opt)
port_esg = np.sum(weights_opt * df_portfolio.esg)
port_vol = portfolio_risk(weights_opt, cov.values)
port_mu = np.sum(weights_opt * df_portfolio.mu / 100)

print(f'✅ ITR: {port_itr:.2f}°C | ESG: {port_esg:.1f} | Vol: {port_vol:.2%} | Mu: {port_mu:.2%}')
df_results.nlargest(10, 'weight_opt')[['ticker', 'NAME', 'SECTOR', 'weight_opt', 'esg', 'itr', 'risk_contrib']].round(3)

## 5. RÉSULTATS + SAUVEGARDE

In [ ]:
# Sauvegarde
df_results.to_csv('../portfolio_final.csv', index=False)
print('💾 portfolio_final.csv')

# Sector weights
sector_weights = df_results.groupby('SECTOR')['weight_opt'].sum().sort_values(ascending=False)
print('\nSecteurs:', sector_weights.round(3))

## 6. VISUALISATIONS INTERACTIVES

In [ ]:
fig = make_subplots(rows=2, cols=2, 
    subplot_titles=('Top 15 Poids', 'Top 15 Risk Contrib', 'ITR vs ESG (size=w)', 'Répartition Sectorielle'),
    specs=[[{}, {}], [{}, {'type':'pie'}]])

# Poids
top_w = df_results.nlargest(15, 'weight_opt')
fig.add_trace(go.Bar(x=top_w.ticker, y=top_w.weight_opt, name='Poids', marker_color='lightblue'), 1, 1)

# Risk
top_r = df_results.nlargest(15, 'risk_contrib')
fig.add_trace(go.Bar(x=top_r.ticker, y=top_r.risk_contrib, name='Risk', marker_color='orange'), 1, 2)

# Scatter
fig.add_trace(go.Scatter(x=df_results.esg, y=df_results.itr, mode='markers',
    marker=dict(size=df_results.weight_opt*3000, color=df_results.SECTOR, colorscale='Viridis', opacity=0.7),
    text=df_results.ticker, hovertemplate='<b>%{text}</b><br>ESG: %{x:.1f}<br>ITR: %{y:.2f}°C<extra></extra>'), 2, 1)

# Pie secteurs
fig.add_trace(go.Pie(labels=sector_weights.index, values=sector_weights.values, name='Secteurs'), 2, 2)

fig.update_layout(height=900, title='🏆 Portefeuille ESG ERC 2°C', showlegend=False)
fig.write_html('../portfolio_visualization.html')
fig.show()

print('💾 portfolio_visualization.html | Ouvrir dans browser')